# 3.8 Ablation (CIFAR-10, PreAct ResNet-18) trên Kaggle

Chạy các dòng của Bảng 5 bằng `experiments/exp38_ablation.py`, rồi gom kết quả bằng
`experiments/exp38_report.py`. Mỗi lần chạy lưu `last.pt` sau từng epoch nên bị ngắt thì chạy lại là tiếp tục.

**Trước khi chạy** (giống `kaggle/README.md`): Accelerator **GPU P100/T4**, **Internet: On**,
Add Input → dataset `Source` của nhóm. Nếu chạy tiếp phiên cũ: Add Input thêm **output của version cũ**.


In [ ]:
# ==== Tham số: chỉnh ở đây ====
VARIANTS = ["erm", "mixup", "input_ac_rp", "ls_0.1", "input_ls_0.2"]   # ưu tiên 1
WDS      = [1e-4, 5e-4]
SEEDS    = [0]
EPOCHS, MILESTONES = 100, "50 75"     # rút gọn từ 200 / "100 150" của bài báo, PHẢI giống nhau cho mọi biến thể
AMP      = False                      # bật/tắt theo kết quả đo của nhóm, nhưng giống nhau cho mọi lần chạy
SMOKE    = True                       # chạy thử 1 epoch x 20 batch cho từng biến thể trước
RUN_FULL = False                      # True: chạy thật (dùng Save & Run All (Commit))
WORKERS  = 4


In [ ]:
import os, sys, glob, shutil, subprocess, time, json
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/core/mixup.py', recursive=True)]
assert cands, "Không thấy Source. Add Input -> chọn dataset chứa thư mục Source"
SRC = '/kaggle/working/Source'
if not os.path.exists(SRC):
    shutil.copytree(cands[0], SRC)
os.chdir(SRC)
DATA, OUT = '/kaggle/working/data', '/kaggle/working/results'
# Chạy tiếp phiên cũ: chép results cũ (nếu có trong input) vào working
for old in glob.glob('/kaggle/input/**/results/exp38_ablation', recursive=True):
    dst = f"{OUT}/exp38_ablation"
    if not os.path.exists(dst):
        shutil.copytree(old, dst)
        print("Đã chép kết quả cũ từ", old)


In [ ]:
!python -m pytest tests -q || python tests/test_core.py

In [ ]:
def run(args):
    cmd = (f"python experiments/exp38_ablation.py --data-dir {DATA} --out-dir {OUT} --workers {WORKERS} "
           f"--epochs {EPOCHS} --milestones {MILESTONES} " + ("--amp " if AMP else "") + args)
    print("$", cmd, flush=True)
    t = time.time()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"-> xong sau {time.time()-t:.0f}s, exit={rc}", flush=True)
    assert rc == 0, "Lệnh lỗi, xem log phía trên"


## 1. Chạy thử từng biến thể (vài phút)

In [ ]:
if SMOKE:
    for v in VARIANTS:
        run(f"--variant {v} --seed 0 --wd 1e-4 --epochs 1 --limit-batches 20 --no-resume --run-name smoke38/{v}")


## 2. Chạy thật

Thứ tự: theo seed, rồi weight decay, rồi biến thể, để nếu hết giờ thì đã có đủ một seed cho mọi ô.
Lần chạy đã xong (có `summary.json`) sẽ được bỏ qua; lần chạy dở sẽ tự resume.

In [ ]:
if RUN_FULL:
    for seed in SEEDS:
        for wd in WDS:
            for v in VARIANTS:
                done = f"{OUT}/exp38_ablation/{v}_wd{wd:g}_seed{seed}/summary.json"
                if os.path.exists(done):
                    print("bỏ qua (đã xong):", done)
                    continue
                run(f"--variant {v} --seed {seed} --wd {wd:g}")
else:
    print("RUN_FULL = False: chỉ chạy thử")


## 3. Bảng so sánh với Bảng 5 + biểu đồ

In [ ]:
if os.path.isdir(f"{OUT}/exp38_ablation"):
    !python experiments/exp38_report.py --results {OUT}
    from IPython.display import Image, display
    for p in sorted(glob.glob(f"{OUT}/exp38_ablation/report_wd*.png")):
        display(Image(p))


In [ ]:
# Đóng gói log (không kèm .pt) để tải từ tab Output. Giữ nguyên thư mục results nếu cần chạy tiếp.
!cd /kaggle/working && zip -qr exp38_logs.zip results -x '*.pt' && ls -lh exp38_logs.zip